# Lab 3: Multi-Step Agent — Chain Reasoning and Actions

**Module 1: Foundations of Agentic AI**  
**Duration:** 30 minutes  
**Difficulty:** Intermediate-Advanced  
**Prerequisites:** Lab 1 (Hello Agent), Lab 2 (Tool-Use Agent)

---

## What You'll Learn

1. Build an **agent loop** that runs until the task is complete
2. Handle multiple sequential tool calls in a single conversation
3. Implement conversation memory that persists across the loop
4. Add simple file-based persistent memory
5. Give the agent 5+ tools and a complex multi-step task

## The Complete Agent Loop

```
          ┌─────────────────────────────┐
          │       USER TASK             │
          └──────────┬──────────────────┘
                     │
                     ▼
          ┌─────────────────────────────┐
     ┌───►│  Send messages to Claude    │
     │    └──────────┬──────────────────┘
     │               │
     │               ▼
     │    ┌─────────────────────────────┐
     │    │  Claude responds            │
     │    └──────────┬──────────────────┘
     │               │
     │        ┌──────┴──────┐
     │        │             │
     │   stop_reason    stop_reason
     │   == tool_use    == end_turn
     │        │             │
     │        ▼             ▼
     │   Execute tool   ┌───────────┐
     │   Append result  │   DONE    │
     │        │         └───────────┘
     └────────┘
```

**The key insight:** The loop continues as long as Claude returns `stop_reason == "tool_use"`. When Claude has everything it needs, it returns `stop_reason == "end_turn"` with the final answer.

---
## Step 1: Setup

In [ ]:
!pip install -q anthropic

In [ ]:
import os
import json
import math
from getpass import getpass
from datetime import datetime

import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5-20241022"

print(f"✓ Ready. Model: {MODEL}")

---
## Step 2: Expanded Tool Suite

A real agent needs multiple tools. Let's give ours five capabilities — enough to handle complex, multi-step tasks.

In [ ]:
# ── Tool Definitions ──────────────────────────────────────────

tools = [
    {
        "name": "calculator",
        "description": "Perform mathematical calculations. Supports arithmetic, exponents, sqrt, trig, and log functions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression to evaluate, e.g. 'sqrt(144) + 2**3'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_current_time",
        "description": "Get the current date and time. Use for timestamps or date-related queries.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "read_file",
        "description": "Read the contents of a text file. Returns the file contents or an error if file doesn't exist.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Path to the file to read"
                }
            },
            "required": ["filename"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a text file. Creates the file if it doesn't exist, overwrites if it does.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Path to the file to write"
                },
                "content": {
                    "type": "string",
                    "description": "Content to write to the file"
                }
            },
            "required": ["filename", "content"]
        }
    },
    {
        "name": "list_files",
        "description": "List all files in the current directory. Use to discover available files.",
        "input_schema": {
            "type": "object",
            "properties": {
                "directory": {
                    "type": "string",
                    "description": "Directory path to list (defaults to current directory)"
                }
            },
            "required": []
        }
    }
]

print(f"✓ Defined {len(tools)} tools: {[t['name'] for t in tools]}")

In [ ]:
# ── Tool Implementations ──────────────────────────────────────

def calculator(expression: str) -> str:
    """Safely evaluate a math expression."""
    allowed = {
        "sqrt": math.sqrt, "abs": abs, "round": round,
        "min": min, "max": max, "pow": pow,
        "pi": math.pi, "e": math.e,
        "sin": math.sin, "cos": math.cos, "tan": math.tan,
        "log": math.log, "log10": math.log10,
    }
    try:
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as e:
        return f"Error: {e}"


def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S (%A)")


def read_file(filename: str) -> str:
    """Read a text file and return its contents."""
    try:
        with open(filename, "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File '{filename}' not found."
    except Exception as e:
        return f"Error reading file: {e}"


def write_file(filename: str, content: str) -> str:
    """Write content to a text file."""
    try:
        with open(filename, "w") as f:
            f.write(content)
        return f"Successfully wrote {len(content)} characters to '{filename}'."
    except Exception as e:
        return f"Error writing file: {e}"


def list_files(directory: str = ".") -> str:
    """List files in a directory."""
    try:
        files = os.listdir(directory)
        if not files:
            return "Directory is empty."
        return "\n".join(sorted(files))
    except Exception as e:
        return f"Error: {e}"


TOOL_FUNCTIONS = {
    "calculator": calculator,
    "get_current_time": get_current_time,
    "read_file": read_file,
    "write_file": write_file,
    "list_files": list_files,
}

print("✓ Tool functions implemented")

---
## Step 3: The Agent Loop

This is the core pattern of every agentic system. The loop:
1. Sends messages to Claude
2. If Claude wants a tool → execute it, append result, loop back
3. If Claude is done → return the final answer

We add a `max_iterations` safety limit to prevent infinite loops.

In [ ]:
def run_agent(
    user_message: str,
    system_prompt: str = "You are a helpful assistant with access to tools. Use them to complete tasks step by step.",
    max_iterations: int = 10,
    verbose: bool = True,
) -> str:
    """Run a multi-step agent loop until the task is complete.
    
    Args:
        user_message: The task for the agent to complete.
        system_prompt: System prompt defining agent behavior.
        max_iterations: Maximum tool-use cycles before stopping.
        verbose: If True, print each step of the agent's reasoning.
    
    Returns:
        The agent's final text response.
    """
    messages = [{"role": "user", "content": user_message}]
    iteration = 0
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"TASK: {user_message}")
        print(f"{'='*60}")
    
    while iteration < max_iterations:
        iteration += 1
        
        if verbose:
            print(f"\n── Step {iteration} ──")
        
        # Call Claude
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )
        
        # Check if we're done
        if response.stop_reason == "end_turn":
            # Extract final text
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            if verbose:
                print(f"\n✓ Agent completed in {iteration} step(s)")
            return final_text
        
        # Process tool calls
        # Claude might return text AND tool_use in the same response
        tool_results = []
        for block in response.content:
            if hasattr(block, "text") and block.text:
                if verbose:
                    print(f"  💭 Thinking: {block.text[:200]}")
            
            if block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input
                tool_use_id = block.id
                
                if verbose:
                    print(f"  🔧 Tool: {tool_name}({json.dumps(tool_input)})")
                
                # Execute the tool
                func = TOOL_FUNCTIONS.get(tool_name)
                if func:
                    try:
                        result = func(**tool_input)
                    except Exception as e:
                        result = f"Error: {e}"
                else:
                    result = f"Error: Unknown tool '{tool_name}'"
                
                if verbose:
                    display = result[:200] + "..." if len(result) > 200 else result
                    print(f"  📋 Result: {display}")
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": result
                })
        
        # Append Claude's response and tool results to conversation
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    
    return "Error: Agent reached maximum iterations without completing the task."


print("✓ Agent loop defined")

---
## Step 4: Test the Agent Loop — Single-Step Task

Let's verify the loop works with a simple task first.

In [ ]:
result = run_agent("What time is it right now?")
print(f"\nFinal answer: {result}")

## Step 5: Multi-Step Task

Now the real test — a task that requires **multiple tools in sequence**.

In [ ]:
result = run_agent(
    "Create a file called 'investment_report.txt' with the following: "
    "1) Today's date and time. "
    "2) If I invest $25,000 at 8% annual compound interest for 10 years, "
    "calculate the final amount and the total interest earned. "
    "3) Also calculate what it would be at 6% and 10% for comparison. "
    "Format it as a nice report."
)

print(f"\n{'='*60}")
print("FINAL RESPONSE:")
print(result)

In [ ]:
# Verify the file was created
if os.path.exists("investment_report.txt"):
    with open("investment_report.txt", "r") as f:
        print("=== Generated Report ===")
        print(f.read())
else:
    print("File was not created. Check the agent output above.")

**What just happened:**
1. Claude called `get_current_time` to get today's date
2. Claude called `calculator` multiple times for each interest rate scenario
3. Claude called `write_file` to save the formatted report
4. Claude returned a final summary

**That's the agent loop in action** — multiple tools, sequential reasoning, autonomous decisions.

---
## Step 6: Adding Persistent Memory

So far our agent has **conversation memory** (the messages list within a single run). But when the function ends, that memory is gone.

Let's add simple **file-based persistent memory** — the agent can save and recall notes across sessions.

In [ ]:
# Add memory tools to our existing tool suite

memory_tools = [
    {
        "name": "save_memory",
        "description": (
            "Save a note to persistent memory. Use this to remember important "
            "information, user preferences, or task results for future reference. "
            "Each memory has a key (topic) and value (the information to remember)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "key": {
                    "type": "string",
                    "description": "A short topic label for this memory, e.g. 'user_name' or 'project_goal'"
                },
                "value": {
                    "type": "string",
                    "description": "The information to remember"
                }
            },
            "required": ["key", "value"]
        }
    },
    {
        "name": "recall_memory",
        "description": (
            "Recall a previously saved memory by its key. Use this to retrieve "
            "information you saved earlier."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "key": {
                    "type": "string",
                    "description": "The topic label of the memory to recall"
                }
            },
            "required": ["key"]
        }
    },
    {
        "name": "list_memories",
        "description": "List all saved memory keys. Use this to see what information has been stored.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

# Combine all tools
all_tools = tools + memory_tools

# ── Memory implementations ──

MEMORY_FILE = "agent_memory.json"

def _load_memory() -> dict:
    """Load the memory store from disk."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return {}

def _save_memory_store(store: dict) -> None:
    """Persist the memory store to disk."""
    with open(MEMORY_FILE, "w") as f:
        json.dump(store, f, indent=2)


def save_memory(key: str, value: str) -> str:
    """Save a key-value pair to persistent memory."""
    store = _load_memory()
    store[key] = {"value": value, "saved_at": datetime.now().isoformat()}
    _save_memory_store(store)
    return f"Saved memory: '{key}'"


def recall_memory(key: str) -> str:
    """Recall a value from persistent memory."""
    store = _load_memory()
    if key in store:
        entry = store[key]
        return f"{entry['value']} (saved at {entry['saved_at']})"
    return f"No memory found for key '{key}'."


def list_memories() -> str:
    """List all memory keys."""
    store = _load_memory()
    if not store:
        return "No memories saved yet."
    return "\n".join(f"- {k}: {v['value'][:50]}..." if len(v['value']) > 50 else f"- {k}: {v['value']}" for k, v in store.items())


# Update the function dispatch map
ALL_TOOL_FUNCTIONS = {
    **TOOL_FUNCTIONS,
    "save_memory": save_memory,
    "recall_memory": recall_memory,
    "list_memories": list_memories,
}

print(f"✓ {len(all_tools)} tools ready (including memory tools)")

## Step 7: Updated Agent Loop with All Tools

In [ ]:
def run_full_agent(
    user_message: str,
    system_prompt: str = (
        "You are an intelligent assistant with tools for calculation, file operations, "
        "and persistent memory. Use tools step by step to complete tasks thoroughly. "
        "Save important results to memory for future reference. "
        "Always verify your work before reporting results."
    ),
    max_iterations: int = 15,
    verbose: bool = True,
) -> str:
    """Run the full agent with all tools including memory.
    
    Args:
        user_message: The task for the agent.
        system_prompt: Agent behavior definition.
        max_iterations: Safety limit for tool-use cycles.
        verbose: Print reasoning steps.
    
    Returns:
        The agent's final response.
    """
    messages = [{"role": "user", "content": user_message}]
    iteration = 0
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"TASK: {user_message[:100]}..." if len(user_message) > 100 else f"TASK: {user_message}")
        print(f"{'='*60}")
    
    while iteration < max_iterations:
        iteration += 1
        
        if verbose:
            print(f"\n── Step {iteration} ──")
        
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system_prompt,
            tools=all_tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            final = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final += block.text
            if verbose:
                print(f"\n✓ Agent completed in {iteration} step(s)")
                print(f"  Tokens used: {response.usage.input_tokens} in / {response.usage.output_tokens} out")
            return final
        
        tool_results = []
        for block in response.content:
            if hasattr(block, "text") and block.text:
                if verbose:
                    print(f"  💭 {block.text[:150]}")
            
            if block.type == "tool_use":
                name = block.name
                inp = block.input
                
                if verbose:
                    print(f"  🔧 {name}({json.dumps(inp)[:100]})")
                
                func = ALL_TOOL_FUNCTIONS.get(name)
                if func:
                    try:
                        result = func(**inp)
                    except Exception as e:
                        result = f"Error: {e}"
                else:
                    result = f"Error: Unknown tool '{name}'"
                
                if verbose:
                    display = result[:150] + "..." if len(result) > 150 else result
                    print(f"  📋 {display}")
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    
    return "Error: Agent hit maximum iteration limit."


print("✓ Full agent loop ready")

---
## Step 8: Complex Multi-Step Task with Memory

Let's give the agent a genuinely complex task that requires planning, multiple tools, and memory.

In [ ]:
result = run_full_agent(
    "I'm planning a savings goal. Here's what I need you to do:\n"
    "1. I want to save $50,000 in 3 years.\n"
    "2. Calculate how much I need to save per month (assuming no interest).\n"
    "3. Now calculate how much per month if I invest in an account earning 5% annually (compounded monthly).\n"
    "4. Calculate the difference — how much I SAVE by investing vs. just saving.\n"
    "5. Save my savings goal and the monthly investment amount to memory.\n"
    "6. Write a summary report to 'savings_plan.txt'.\n"
    "7. Give me a final summary."
)

print(f"\n{'='*60}")
print("FINAL ANSWER:")
print(result)

In [ ]:
# Verify memory was saved
print("=== Saved Memories ===")
print(list_memories())

# Verify file was written
if os.path.exists("savings_plan.txt"):
    print("\n=== Savings Plan Report ===")
    print(read_file("savings_plan.txt"))

## Step 9: Memory Recall Across "Sessions"

Now let's simulate a new session — the agent starts fresh but can recall saved memories.

In [ ]:
# Simulate a "new session" — no conversation history, but memory persists
result = run_full_agent(
    "Check your memory — do you have any saved information about my savings goals? "
    "If so, tell me what my goals are and calculate how much I would have "
    "after 5 years instead of 3, at the same monthly investment amount."
)

print(f"\n{'='*60}")
print("ANSWER:")
print(result)

**This is what makes agents powerful:**
- Conversation memory lets them reason across multiple steps within a task
- Persistent memory lets them carry knowledge across sessions
- Tool-use lets them act on the world, not just talk about it

---
## Exercises

### Exercise 1: Research Agent

Give the agent this task: *"List all files in the current directory. Read any .txt files you find. Summarize what you learned and save a summary to memory."*

This tests multi-step reasoning: list → read → analyze → save.

In [ ]:
# YOUR CODE HERE



### Exercise 2: Add Iteration Tracking

Modify `run_full_agent` to return not just the final answer, but also:
- Total number of iterations
- List of tools used (in order)
- Total tokens consumed (input + output)

Return these as a dictionary alongside the answer.

In [ ]:
# YOUR CODE HERE



### Exercise 3: Guardrails

Add a safety check to the agent loop that:
1. Blocks `write_file` if the filename contains `..` (path traversal)
2. Limits file writes to files under 10,000 characters
3. Logs every tool call to a list that can be audited

This is a preview of the **evaluation and safety** patterns from Section 4 of the curriculum.

In [ ]:
# YOUR CODE HERE



---
## Key Takeaways

1. **The agent loop is the core pattern** — send → check stop_reason → execute tools → loop
2. **Conversation memory is the message list** — every API call replays the full history
3. **Persistent memory uses files/databases** — survives across agent sessions
4. **Safety limits are critical** — max_iterations prevents infinite loops, guardrails prevent harm
5. **Complex tasks emerge from simple tools** — 5 basic tools can handle sophisticated workflows

## You've Built a Real Agent!

Across three labs, you've gone from zero to a working multi-step agent with:
- ✅ LLM brain (Claude API)
- ✅ Tool-use (5+ tools with autonomous selection)
- ✅ Memory (conversation + persistent file-based)
- ✅ Planning (multi-step task decomposition)
- ✅ The agent loop (perceive → reason → act → observe → repeat)

## What's Next: Capstone Project

Use everything you've learned to build a **Personal Research Agent** — see `capstone_starter.ipynb` and `capstone.md` for the project brief.

---
*Lab 3 Complete — Module 1: Foundations of Agentic AI*

---
## Cleanup

Run this cell to clean up files created during the lab.

In [ ]:
# Optional: Clean up generated files
for f in ["investment_report.txt", "savings_plan.txt", "agent_memory.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

print("✓ Cleanup complete")